# 06 — Experimentos de Machine Learning

Compara dois experimentos de Regressão Logística: Modelo A (contexto) e Modelo B (contexto + dinâmica da ocorrência). Treino 2020–2023, validação 2024 e 2025 reservado.

In [ ]:
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
import joblib

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
PASTA_MODELOS = os.path.join(PASTA_PROJETO, "outputs", "modelos")
os.makedirs(PASTA_MODELOS, exist_ok=True)

df = pd.read_parquet(os.path.join(PASTA_PROCESSED, "prf_acidentes_silver_2020_2025.parquet"))
df["br_categoria"] = df["br"].astype("Int64").astype("string").fillna("Não informado")
TARGET = "acidente_grave"

In [ ]:
colunas_tracado = [c for c in df.columns if c.startswith("tracado_") and c != "tracado_via"]

categoricas_a = ["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo"]
numericas_a = ["km","mes","hora","fim_semana","qtd_caracteristicas_tracado"] + colunas_tracado
features_a = categoricas_a + numericas_a

categoricas_b = categoricas_a + ["tipo_acidente"]
numericas_b = numericas_a + ["veiculos"]
features_b = categoricas_b + numericas_b

proibidas = {"mortos","feridos_graves","feridos_leves","feridos","ilesos","ignorados","pessoas","classificacao_acidente","acidente_fatal"}
assert not any(c in proibidas for c in features_a + features_b)

treino = df[df["ano"] <= 2023].copy()
validacao = df[df["ano"] == 2024].copy()
teste_final = df[df["ano"] == 2025].copy()
print(len(treino), len(validacao), len(teste_final))

In [ ]:
def encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse=True)

def preprocessador(categoricas, numericas):
    return ColumnTransformer([
        ("categoricas", encoder(), categoricas),
        ("numericas", StandardScaler(), numericas)
    ])

def avaliar(modelo, X, y, nome):
    pred = modelo.predict(X)
    prob = modelo.predict_proba(X)[:,1]
    return {
        "modelo": nome,
        "accuracy": accuracy_score(y,pred),
        "precision": precision_score(y,pred),
        "recall": recall_score(y,pred),
        "f1": f1_score(y,pred),
        "roc_auc": roc_auc_score(y,prob),
        "pr_auc": average_precision_score(y,prob)
    }

In [ ]:
y_train = treino[TARGET]
y_val = validacao[TARGET]

modelo_a = Pipeline([
    ("preprocessamento", preprocessador(categoricas_a, numericas_a)),
    ("modelo", LogisticRegression(solver="saga", max_iter=500, random_state=42))
])
modelo_a.fit(treino[features_a], y_train)
metricas_a = avaliar(modelo_a, validacao[features_a], y_val, "Modelo A - Contexto")

modelo_b = Pipeline([
    ("preprocessamento", preprocessador(categoricas_b, numericas_b)),
    ("modelo", LogisticRegression(solver="saga", max_iter=500, random_state=42))
])
modelo_b.fit(treino[features_b], y_train)
metricas_b = avaliar(modelo_b, validacao[features_b], y_val, "Modelo B - Contexto + dinâmica")

comparacao = pd.DataFrame([metricas_a, metricas_b])
display(comparacao)
comparacao.to_csv(os.path.join(PASTA_MODELOS, "baseline_logistic_regression.csv"), index=False, encoding="utf-8-sig")
joblib.dump(modelo_a, os.path.join(PASTA_MODELOS, "logistica_modelo_a.joblib"))
joblib.dump(modelo_b, os.path.join(PASTA_MODELOS, "logistica_modelo_b.joblib"))
print("2025 continua reservado.")